In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, ReLU, GlobalAveragePooling2D, Dense, Input,
    Add, MaxPooling2D
)
from sklearn.utils import shuffle
from pathlib import Path

# --- CONFIGURATION ---
train_csv = Path("/path/to/training/data.csv")
input_shape = (128, 128, 3)
batch_size = 32
epochs = 50
save_path = "resnext2d_regression_model_Q1.keras"

# --- ResNeXt Block ---
def resnext_block(x, filters, groups=32):
    shortcut = x  # Save for skip connection
    in_channels = x.shape[-1]

    # 1x1 Conv to reduce depth
    x = Conv2D(filters, (1, 1), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    # 3x3 Grouped Conv
    x = Conv2D(filters, (3, 3), padding='same', groups=groups, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    # 1x1 Conv to restore depth
    x = Conv2D(filters, (1, 1), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)

    # Match shortcut dimensions using 1x1 Conv if needed
    if in_channels != filters:
        shortcut = Conv2D(filters, (1, 1), padding='same', use_bias=False)(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add skip connection
    x = Add()([x, shortcut])
    x = ReLU()(x)
    return x

# --- ResNeXt-Style Regression Model ---
def build_resnext_regression(input_shape=(128, 128, 3), num_blocks=3, filters=64, groups=8):
    inputs = Input(shape=input_shape)

    x = Conv2D(filters, (7, 7), strides=2, padding='same', use_bias=False)(inputs)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = MaxPooling2D(pool_size=(3, 3), strides=2, padding='same')(x)

    for i in range(num_blocks):
        x = resnext_block(x, filters, groups=groups)
        filters *= 2  # Increase filter size after each block

    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dense(128, activation='relu')(x)
    outputs = Dense(1, activation='linear')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# --- Custom Data Generator (same as previous models) ---
class SpectrogramDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, batch_size, input_shape, shuffle=True):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.input_shape = input_shape
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch = self.df.iloc[index * self.batch_size:(index + 1) * self.batch_size]
        X = np.zeros((len(batch), *self.input_shape), dtype=np.float32)
        y = np.zeros((len(batch),), dtype=np.float32)

        for i, (_, row) in enumerate(batch.iterrows()):
            try:
                arr = np.load(row["filepath"]).astype(np.float32)
                arr = tf.image.resize(arr[..., np.newaxis], self.input_shape[:2]).numpy()
                arr = np.repeat(arr, 3, axis=-1)  # Convert grayscale to RGB
                X[i] = arr
                y[i] = row["label"]
            except Exception as e:
                print(f"⚠️ Error loading {row['filepath']}: {e}")

        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            self.df = shuffle(self.df)

# --- Load Training Data ---
train_df = pd.read_csv(train_csv)
train_gen = SpectrogramDataGenerator(train_df, batch_size, input_shape)

# --- Build and Train ResNeXt Model ---
model = build_resnext_regression(input_shape=input_shape)
model.summary()

checkpoint = tf.keras.callbacks.ModelCheckpoint(save_path, save_best_only=False)

model.fit(
    train_gen,
    epochs=epochs,
    callbacks=[checkpoint],
    verbose=1
)

print(f"\n✅ ResNeXt-2D regression model trained and saved as: {save_path}")


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

# --- CONFIG ---
model_path = Path("/path/to/trained/resnext2d_regression_model_Q1.keras")
val_csv = Path("/path/to/validation/data.csv")
input_shape = (128, 128, 3)  # Match training input shape

# --- Load trained model ---
model = tf.keras.models.load_model(model_path, compile=False)

# --- Load validation data ---
val_df = pd.read_csv(val_csv)

# --- Prepare input and labels ---
X_val = np.zeros((len(val_df), *input_shape), dtype=np.float32)
y_true = np.zeros((len(val_df),), dtype=np.float32)

for i, (_, row) in enumerate(val_df.iterrows()):
    arr = np.load(row["filepath"]).astype(np.float32)

    # Add channel and resize
    arr = tf.image.resize(arr[..., np.newaxis], input_shape[:2]).numpy()
    arr = np.repeat(arr, 3, axis=-1)  # Convert to 3-channel RGB-like
    X_val[i] = arr
    y_true[i] = row["label"]

# --- Predict ---
y_pred = model.predict(X_val, batch_size=32).flatten()

# --- Compute squared errors and MSE ---
squared_errors = (y_pred - y_true) ** 2
mse = np.mean(squared_errors)

print("\n✅ Squared error for each sample:")
print(squared_errors)

print(f"\n📊 Mean Squared Error (Validation): {mse:.6f}")


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

# --- CONFIG ---
model_path = Path("/path/to/trained/resnext2d_regression_model_Q1.keras")
val_csv = Path("/path/to/training/data.csv")
input_shape = (128, 128, 3)  # Update if your spectrograms have different shape

# --- Load model (no compile needed just for prediction) ---
model = tf.keras.models.load_model(model_path, compile=False)

# --- Load validation data ---
val_df = pd.read_csv(val_csv)

# --- Preprocess all validation data ---
X_val = np.zeros((len(val_df), *input_shape), dtype=np.float32)
y_true = np.zeros((len(val_df),), dtype=np.float32)

for i, (_, row) in enumerate(val_df.iterrows()):
    arr = np.load(row["filepath"]).astype(np.float32)

    # Add channel dimension and resize if needed
    arr = tf.image.resize(arr[..., np.newaxis], input_shape[:2]).numpy()

    X_val[i] = arr
    y_true[i] = row["label"]

# --- Predict with the model ---
y_pred = model.predict(X_val, batch_size=32)

# --- Compute Squared Error and MSE ---
squared_errors = (y_pred.flatten() - y_true) ** 2
mse = np.mean(squared_errors)

print("\n✅ Squared error for each training sample:")
print(squared_errors)

print(f"\n📊 Mean Squared Error (Training): {mse:.6f}")


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

# --- CONFIGURATION ---
model_path = Path("/path/to/trained/resnext2d_regression_model_Q1.keras")
test_csv = Path("/path/to/testing/data.csv")
input_shape = (128, 128, 3)  # Modify if your spectrogram shape is different

# --- Load Model (no need to compile for prediction) ---
model = tf.keras.models.load_model(model_path, compile=False)

# --- Load Test Data ---
test_df = pd.read_csv(test_csv)

# --- Preprocess Test Data ---
X_test = np.zeros((len(test_df), *input_shape), dtype=np.float32)
y_true = np.zeros((len(test_df),), dtype=np.float32)

for i, (_, row) in enumerate(test_df.iterrows()):
    arr = np.load(row["filepath"]).astype(np.float32)

    # Ensure correct shape and resize if necessary
    arr = tf.image.resize(arr[..., np.newaxis], input_shape[:2]).numpy()
    X_test[i] = arr
    y_true[i] = row["label"]

# --- Predict with the Model ---
y_pred = model.predict(X_test, batch_size=32).flatten()

# --- Compute Root Mean Squared Error ---
mse = np.mean((y_pred - y_true) ** 2)
rmse = np.sqrt(mse)

print(f"\n📊 Root Mean Squared Error (Test): {rmse:.6f}")


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

# --- CONFIGURATION ---
model_path = Path("/path/to/trained/resnext2d_regression_model_Q1.keras")
test_csv = Path("/path/to/testing/data.csv")
input_shape = (128, 128, 3)  # Modify if your spectrogram shape is different

# --- Load Model (no need to compile for prediction) ---
model = tf.keras.models.load_model(model_path, compile=False)

# --- Load Test Data ---
test_df = pd.read_csv(test_csv)

# --- Preprocess Test Data ---
X_test = np.zeros((len(test_df), *input_shape), dtype=np.float32)
y_true = np.zeros((len(test_df),), dtype=np.float32)

for i, (_, row) in enumerate(test_df.iterrows()):
    arr = np.load(row["filepath"]).astype(np.float32)

    # Ensure correct shape and resize if necessary
    arr = tf.image.resize(arr[..., np.newaxis], input_shape[:2]).numpy()
    X_test[i] = arr
    y_true[i] = row["label"]

# --- Predict with the Model ---
y_pred = model.predict(X_test, batch_size=32).flatten()

# --- Compute RMSE ---
mse = np.mean((y_pred - y_true) ** 2)
rmse = np.sqrt(mse)

# --- Compute Coefficient of Determination (R²) ---
ss_res = np.sum((y_true - y_pred) ** 2)
ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
r2 = 1 - (ss_res / ss_tot)

# --- Print Results ---
print(f"\n📊 Root Mean Squared Error (Test): {rmse:.6f}")
print(f"📈 Coefficient of Determination (R²): {r2:.6f}")
